# MR2b — Type Count Store MapReduce
**Input**: `manhattan_liquor_stores_geocoded.csv`
**Output**: `type_count_store_mr.csv`

Schema: `zone_id, method_of_operation, count`

In [1]:
import h3
import pandas as pd
from collections import defaultdict

INPUT  = 'manhattan_liquor_stores_geocoded.csv'
OUTPUT = 'type_count_store_mr.csv'
H3_RES = 10

df = pd.read_csv(INPUT)
df['latitude']  = pd.to_numeric(df['latitude'],  errors='coerce')
df['longitude'] = pd.to_numeric(df['longitude'], errors='coerce')
df = df.dropna(subset=['latitude','longitude'])
df['method_of_operation'] = df['method_of_operation'].fillna('Unknown').str.strip().str.title()

print(f'Input rows: {len(df):,}')
print('Unique methods:', df['method_of_operation'].nunique())

Input rows: 6,925
Unique methods: 1177


In [2]:
# ── MAP ───────────────────────────────────────────────────────────────────────
# Emit: key=(zone_id, method_of_operation), val=1

def mapper(row):
    try:
        zone_id = h3.latlng_to_cell(float(row['latitude']), float(row['longitude']), H3_RES)
    except:
        return None, None
    key = (zone_id, str(row['method_of_operation']))
    return key, 1

mapped = [mapper(row) for _, row in df.iterrows()]
mapped = [(k,v) for k,v in mapped if k is not None]
print(f'Mapped pairs: {len(mapped):,}')

Mapped pairs: 6,925


In [3]:
# ── REDUCE ────────────────────────────────────────────────────────────────────
# Sum count per (zone_id, method_of_operation)

acc = defaultdict(int)
for key, val in mapped:
    acc[key] += val

rows = [
    {'zone_id': zone_id, 'method_of_operation': method, 'count': cnt}
    for (zone_id, method), cnt in acc.items()
]

# ── OUTPUT ────────────────────────────────────────────────────────────────────
out = pd.DataFrame(rows).sort_values(['zone_id','count'], ascending=[True,False]).reset_index(drop=True)
out.to_csv(OUTPUT, index=False)

print(f'Output rows : {len(out):,}')
print(f'Unique zones: {out["zone_id"].nunique():,}')
print(f'Saved → {OUTPUT}')

# Sample: top methods per zone
print('\nSample zone breakdown:')
sample_zone = out['zone_id'].iloc[0]
print(out[out['zone_id']==sample_zone].to_string(index=False))

Output rows : 6,576
Unique zones: 1,823
Saved → type_count_store_mr.csv

Sample zone breakdown:
        zone_id                            method_of_operation  count
8a2a1008804ffff      Tavern Serving Beer Cider Wine And Liquor      1
8a2a1008804ffff Restaurant Serving Beer, Wine, Liquor, & Cider      1
8a2a1008804ffff  Restaurant Serving Beer Cider Wine And Liquor      1
